# spi_nb_bronze_tax

**Purpose.** Bronze ingestion of the AEAT tax revenue XLSX into the Lakehouse,
for the Tax source (source 5 of `spi_pl_bronze`, per ADR-007).

**Inputs.** One XLSX workbook, `Ingresos_por_Delegaciones.xlsx`, from
`https://sede.agenciatributaria.gob.es/static_files/Sede/Tema/Estadisticas/Recaudacion_Tributaria/Informes_mensuales/`.
Only sheet `datos_delegaciones` is read. No runtime parameters.

**Outputs.** Delta table `spi_bronze_tax_raw` in the default Lakehouse,
written with `mode="overwrite"`. The table holds the raw grid of the sheet
(ADR-013, 2026-10-03 addendum): positional columns `col_01` … `col_76`, with
the header row kept as the first row and the trailing empty rows kept as
received. Every row carries `_ingestion_timestamp` and `_source_file`; all
columns are stored as string, per the Bronze layer principle.

**Dependencies.** `pandas`, `requests`, `openpyxl`, `deltalake`. Runs on the
Fabric **Python** runtime (no Spark session), per ADR-012.

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>}`. On failure, the exception propagates uncaught; a
chained Script activity reads the Notebook activity's status, per the
2026-09-30 addendum to ADR-009.

**Last updated.** 2026-10-03.

In [ ]:
from datetime import datetime, timezone
import io
import json
import logging

import pandas as pd
import requests
from deltalake import write_deltalake
from openpyxl import load_workbook

# ────────────────────────────────────────────────────────────────────────────
# Configure environment variables
# ────────────────────────────────────────────────────────────────────────────

URL_XLSX = (
    "https://sede.agenciatributaria.gob.es/static_files/Sede/Tema/Estadisticas/"
    "Recaudacion_Tributaria/Informes_mensuales/Ingresos_por_Delegaciones.xlsx"
)
SHEET_NAME = "datos_delegaciones"

TIMEOUT = 60

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("tax_xlsx_bronze_load")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function download_file
# ────────────────────────────────────────────────────────────────────────────

def download_file() -> bytes:
    """
    Download the AEAT XLSX file into memory.
    Returns:
        bytes: Content of the downloaded XLSX file.
    """
    file_name = URL_XLSX.split("/")[-1]
    log.info("Downloading: %s", file_name)
    response = requests.get(URL_XLSX, timeout=TIMEOUT)
    response.raise_for_status()
    log.info(
        "File downloaded: %s  |  Bytes: %s",
        file_name, len(response.content),
    )
    return response.content

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function cell_to_str
# ────────────────────────────────────────────────────────────────────────────

def cell_to_str(value) -> str:
    """Convert a openpyxl cell to string. (Bronze layer)"""
    if value is None:
        return ''
    return str(value)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function parse_tax_xlsx
# ────────────────────────────────────────────────────────────────────────────

def parse_tax_xlsx(content: bytes, file_name: str) -> pd.DataFrame:
    """
    Read the XLSX file content and return a DataFrame
    Args:
        content (bytes): Content of the downloaded XLSX file.
        file_name (str): Name of the source file (from URL_XLSX).
    Returns:
        pd.DataFrame: A Dataframe with data and 2 lineage columns
        (_ingestion_timestamp & _source_file)
    """
    wb = load_workbook(io.BytesIO(content), read_only=True, data_only=True)
    sh = wb[SHEET_NAME]
    all_rows = []
    for row in sh.iter_rows(values_only=True):
        all_rows.append([cell_to_str(v) for v in row])
    col_names = [f"col_{i:02d}" for i in range(1, len(all_rows[0]) + 1)]
    df = pd.DataFrame(all_rows, columns=col_names)
    df["_ingestion_timestamp"] = datetime.now(timezone.utc).isoformat()
    df["_source_file"] = file_name
    log.info(
        "Parsed: %s  |  Rows: %s  |  Cols: %s",
        file_name, df.shape[0], df.shape[1],
    )
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> int:
    """
    Main function that executes the entire process Data loading from AEAT URL.
    Returns:
        int: Number of rows written to the Bronze table.
    """
    content = download_file()
    df = parse_tax_xlsx(content, URL_XLSX.split("/")[-1])

    # Header row and trailing empty rows are kept as received (ADR-013, 2026-10-03 addendum)
    table_uri = (
        "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
        "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables/spi_bronze_tax_raw"
    )
    storage_options = {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }
    write_deltalake(table_uri, df, mode="overwrite", storage_options=storage_options)
    log.info(f"Data stored in spi_bronze_tax_raw: {len(df)} rows")
    return len(df)

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to scr_log_tax (ADR-009).

rows_written = main()
notebookutils.notebook.exit(json.dumps({"rows_written": rows_written}))